# Trigonometric Functions

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.trigonometric_catalog import (
    arcsine_example, arctangent_example, trigonometric_example_graph,
)
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

def show_signature(example):
    name, signature, _ = example
    display(SVG(field_signature_svg([(name, signature, {})])))

def show_calls(example):
    graph = trigonometric_example_graph(example)
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    print("Source-declared leaves", graph.leaf_counts)

## `ArcSin`
Compute the arcsine of a fixed-point number.

Implements the unitary:
$$
    |a\rangle|0\rangle \rightarrow |a\rangle|\arcsin(a)\rangle
$$

#### Parameters
 - `bitsize`: Number of bits used to represent the number.
 - `num_frac`: Number of fraction bits in the number.
 - `num_iters`: Number of Newton-Raphson iterations. Defaults to 4; the reference studies 3, 4, or 5 iterations.
 - `degree`: Degree of the polynomial of the initial approximation. Defaults to 4; the reference studies degree-3, 4, 5, or 6 polynomials. 

#### Registers
 - `x`: `bitsize`-sized input register.
 - `result`: `bitsize`-sized output register. 

#### References
 - [Optimizing Quantum Circuits for Arithmetic](https://arxiv.org/abs/1805.12445). Appendix D.


The locked scalar callback interprets the input at full-width fractional precision and XORs an unmasked result. Its signature records the requested fractional width for the resource declaration. The independent Appendix D circuit has a separate fixed-point input and output contract.

In [ ]:
from flagquantum.simulation.classical_arcsine import source_arcsine_classical_values

### Example Instances

In [ ]:
arcsin = arcsine_example()
arcsin

#### Graphical Signature

In [ ]:
show_signature(arcsin)

### Call Graph

In [ ]:
show_calls(arcsin)

## Source word behavior and native Appendix D lowering

The original ten-bit callback is evaluated independently. At four bits the valid word 15 produces the integer 19 before any register encoding. The native circuit uses an integer bit plus fractional bits, an explicit approximation polynomial and an explicit half-pi word. The following binding reports the emitted circuit only. Larger Appendix D registers are verified by separate sparse and branch-certificate tests.

In [ ]:
from collections import Counter
from math import pi, sqrt
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.compiler.registers.fixed_point.arcsine_coefficients import arcsine_polynomial_coefficients
from flagquantum.compiler.registers.fixed_point.fixed_point_arcsine import (
    fixed_point_arcsine_ir, fixed_point_arcsine_requirements,
)
from flagquantum.dynamic import DynamicCircuit

x = int(1024 / sqrt(2))
assert source_arcsine_classical_values(x, 0, 10) == (x, 804)
assert source_arcsine_classical_values(15, 0, 4) == (15, 19)

coefficients, error = arcsine_polynomial_coefficients(2, degree=1)
work, slots = fixed_point_arcsine_requirements(2, 2, 5, 1, precision=2, iterations=1)
gates = fixed_point_arcsine_ir(
    (0, 1, 2), (3, 4, 5, 6, 7), tuple(range(8, 8 + work)), tuple(range(slots)),
    coefficients=coefficients, polynomial_fraction_bits=2,
    half_pi_word=int((pi / 2) * 16), precision=2, iterations=1,
)
circuit = DynamicCircuit(8 + work)
circuit.append(FieldOperation(8 + work, gates))
print("Independent polynomial factor error bound", float(error))
print("Concrete Appendix D emitted primitives", dict(Counter(g.name for g in circuit.to_ir().instructions)))
print("Clean work and reusable feedback slots", work, slots)

## ArcTan lookup and source estimate

The original notebook only illustrates ArcSin. This extension keeps the same family in view. The source ArcTan's symbolic product model and the bounded native QROM are separate. On the native path, the sign and magnitude are XORed into occupied targets and the same operation restores them. The state check includes all inputs, all target values and every work wire in the sampled trajectories.

In [ ]:
from math import atan
from sympy import Symbol
import torch
from flagquantum.compiler.registers.fixed_point.trigonometric_resource_model import (
    ARCTANGENT_PRODUCT_CALLS, source_arctangent_t_count,
)
from flagquantum.compiler.registers.lookup.arctan_lookup import arctan_lookup_ir
from flagquantum.runtime.dynamic.execution import run_dynamic

print("Source symbolic T model", source_arctangent_t_count(Symbol("n"), ARCTANGENT_PRODUCT_CALLS))
print("Source declaration", arctangent_example(3, Symbol("n"))[2])
gates = arctan_lookup_ir((0, 1, 2), 3, (4, 5, 6), (7, 8), (0, 1))
op = FieldOperation(9, gates)
state = torch.zeros(512, dtype=torch.complex128)
expected = torch.zeros_like(state)
labels = torch.arange(128, dtype=torch.float64)
amplitudes = torch.complex(labels + 1, 3 * labels - 5)
amplitudes /= amplitudes.norm()
for i, amplitude in enumerate(amplitudes):
    select = i >> 4
    flags = (int(select != 0) << 3) | int(16 * atan(select) / pi)
    state[i << 2] = expected[(i ^ flags) << 2] = amplitude
circuit = DynamicCircuit(9, inputs=state, dtype=torch.complex128)
circuit.append(op)
actual = run_dynamic(circuit, shots=8, seed=31, strategy="trajectory")
torch.testing.assert_close(actual.final_states.reshape(8, -1), expected.expand(8, -1), atol=1e-12, rtol=0)
circuit.append(op)
restored = run_dynamic(circuit, shots=8, seed=31, strategy="trajectory")
torch.testing.assert_close(restored.final_states.reshape(8, -1), state.expand(8, -1), atol=1e-12, rtol=0)
print("Concrete lookup emitted primitives", dict(Counter(g.name for g in gates)))